Write a query that calculates the difference between the highest salaries found in the marketing and engineering departments. Output just the absolute difference in salaries.

In [0]:
%skip
%sql
CREATE TABLE ska_catalog.bronze.db_employee (id INT,first_name VARCHAR(50),last_name VARCHAR(50),salary INT,department_id INT);

INSERT INTO ska_catalog.bronze.db_employee (id, first_name, last_name, salary, department_id) VALUES(10306, 'Ashley', 'Li', 28516, 4),(10307, 'Joseph', 'Solomon', 19945, 1),(10311, 'Melissa', 'Holmes', 33575, 1),(10316, 'Beth', 'Torres', 34902, 1),(10317, 'Pamela', 'Rodriguez', 48187, 4),(10320, 'Gregory', 'Cook', 22681, 4),(10324, 'William', 'Brewer', 15947, 1),(10329, 'Christopher', 'Ramos', 37710, 4),(10333, 'Jennifer', 'Blankenship', 13433, 4),(10339, 'Robert', 'Mills', 13188, 1);

CREATE TABLE ska_catalog.bronze.db_dept (id INT,department VARCHAR(50));

INSERT INTO ska_catalog.bronze.db_dept (id, department) VALUES(1, 'engineering'),(2, 'human resource'),(3, 'operation'),(4, 'marketing');

In [0]:
%sql
SELECT * FROM ska_catalog.bronze.db_employee

In [0]:
%sql
SELECT * FROM ska_catalog.bronze.db_dept;

In [0]:
%sql
-- Write a query that calculates the difference between the highest salaries found in the marketing and engineering departments. Output just the absolute difference in salaries.
SELECT
  MAX(CASE WHEN d.department = 'marketing' THEN e.salary END)
  -
  MAX(CASE WHEN d.department = 'engineering' THEN e.salary END)
  AS `salary_diff`
FROM ska_catalog.bronze.db_employee e
JOIN ska_catalog.bronze.db_dept d
ON e.department_id = d.id

In [0]:
from pyspark.sql.functions import max, when

df_emp = spark.sql("SELECT * FROM ska_catalog.bronze.db_employee")
df_dept = spark.sql("SELECT * FROM ska_catalog.bronze.db_dept")

df_joined = df_emp.join(df_dept, df_emp.department_id == df_dept.id)

df_agg = df_joined.agg(
    (max(when(df_dept.department == 'marketing', df_emp.salary)) -
     max(when(df_dept.department == 'engineering', df_emp.salary))
    ).alias("salary_diff")
)

display(df_agg)